In [1]:
!pip install -q -U transformers accelerate bitsandbytes sacrebleu huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [2]:
import transformers, huggingface_hub, accelerate, bitsandbytes, sacrebleu
print("transformers", transformers.__version__)
print("huggingface_hub", huggingface_hub.__version__)
print("all imports OK")

transformers 5.23.0
huggingface_hub 2.4.0
all imports OK


In [4]:
from huggingface_hub import list_repo_files
files = list_repo_files("google/IndicGenBench_flores_in", repo_type="dataset")
print([f for f in files if "_or_" in f or "_hi_" in f])

['flores_en_hi_dev.json', 'flores_en_hi_test.json', 'flores_en_or_dev.json', 'flores_en_or_test.json', 'flores_hi_en_dev.json', 'flores_hi_en_test.json', 'flores_or_en_dev.json', 'flores_or_en_test.json']


In [5]:
from huggingface_hub import hf_hub_download
import json

def get(fname):
    p = hf_hub_download("google/IndicGenBench_flores_in", fname, repo_type="dataset")
    return json.load(open(p))

en_or = get("flores_en_or_test.json")
en_hi = get("flores_en_hi_test.json")

print(en_or.keys())
print(len(en_or["examples"]), len(en_hi["examples"]))
print(en_or["examples"][0])
print(en_hi["examples"][0])

flores_en_or_test.json:   0%|          | 0.00/639k [00:00<?, ?B/s]

flores_en_hi_test.json:   0%|          | 0.00/614k [00:00<?, ?B/s]

dict_keys(['canary', 'examples'])
1012 1012
{'source': 'Taiwan beginning start way back in 15th century where European sailors passing by record the island’s name as Ilha Formosa, or beautiful island.', 'target': 'ତାଇୱାନ 15 ତମ ଶତାବ୍ଦୀର ଆରମ୍ଭରେ ଆରମ୍ଭ କରିଥିଲା ଯେଉଁଠାରେ ୟୁରୋପୀୟ ନାବିକମାନେ ଦ୍ବୀପପୁଞ୍ଜର ନାମ ଇଲହା ଫର୍ମୋସା କିମ୍ବା ସୁନ୍ଦର ଦ୍ବୀପ ଭାବରେ ରେକର୍ଡ କରିଥିଲେ।', 'lang': 'or', 'translation_direction': 'enxx'}
{'source': 'After a week of losses in the midterm election, Bush told an audience about the expansion of trade in Asia.', 'target': 'मध्यावधि चुनाव में एक हफ्ते तक हारने के बाद, बुश ने लोगों को एशिया में व्यापार के विस्तार के बारे में बताया।', 'lang': 'hi', 'translation_direction': 'enxx'}


In [9]:
import random

or_map = {e["source"].strip(): e["target"] for e in en_or["examples"]}
hi_map = {e["source"].strip(): e["target"] for e in en_hi["examples"]}
common = sorted(set(or_map) & set(hi_map))
print("Odia file:", len(or_map), " Hindi file:", len(hi_map), " shared English sentences:", len(common))

Odia file: 1012  Hindi file: 1012  shared English sentences: 1012


In [11]:
from google.colab import drive
import os
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/odia_eval"
os.makedirs(OUT, exist_ok=True)
print("saving to", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saving to /content/drive/MyDrive/odia_eval


In [12]:
import json
json.dump({"english": english, "odia": odia, "hindi": hindi},
          open(f"{OUT}/sample200.json", "w", encoding="utf-8"), ensure_ascii=False)

print(len(english), len(odia), len(hindi))
print(english[0]); print(odia[0]); print(hindi[0])

200 200 200
These couples may choose to make an adoption plan for their baby.
ଏହି ଦମ୍ପତ୍ତିମାନେ ସେମାନଙ୍କ ସନ୍ତାନ ପାଇଁ ଏକ ପୋଷ୍ୟ ଶିଶୁ ଗ୍ରହଣ କରିବା ଯୋଜନା ବାଛିପାରନ୍ତି।
ये जोड़े अपने बच्चे के लिए गोद लेने की योजना बनाने का विकल्प चुन सकते हैं.


In [14]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("google/gemma-4-E4B-it")

def fertility(sents):
    t = sum(len(tok.encode(s, add_special_tokens=False)) for s in sents)
    w = sum(len(s.split()) for s in sents)
    return round(t / w, 2)

print("Gemma  Odia:", fertility(odia), " Hindi:", fertility(hindi), " English:", fertility(english))

Gemma  Odia: 4.88  Hindi: 1.39  English: 1.24


In [15]:
def tok_per_char(sents):
    return round(sum(len(tok.encode(s, add_special_tokens=False)) for s in sents) / sum(len(s) for s in sents), 3)

print("tokens/char  Odia:", tok_per_char(odia), " Hindi:", tok_per_char(hindi), " English:", tok_per_char(english))

tokens/char  Odia: 0.708  Hindi: 0.273  English: 0.205


In [16]:
import json
fert = {"gemma-4-E4B-it": {"tokens_per_word": {"odia": 4.88, "hindi": 1.39, "english": 1.24},
                           "tokens_per_char": {"odia": 0.708, "hindi": 0.273, "english": 0.205}}}
json.dump(fert, open(f"{OUT}/fertility.json", "w"), indent=2)

In [17]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-4-E4B-it", device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                           bnb_4bit_compute_dtype=torch.float16))

def ask(prompt, max_new_tokens=512):
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    new = out[0][ids["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), len(new)

print(ask("Translate into Odia. Output only the translation.\n\n" + english[0]))

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

('ଏହି ଯୋଡ଼ିକାଙ୍କର ନିଜର ସନ୍ତାନଙ୍କ ପାଇଁ ଗୋଟିଏ ଗୃహୀକରଣ ଯୋଜନା କରିବାକୁ ଚାହିଁ ରହିଥାନ୍ତି।', 50)


In [19]:
import sacrebleu
hyps, lens = [], []
for s in english[:20]:
    h, n = ask("Translate the following English sentence into Odia. Output only the translation.\n\n" + s)
    hyps.append(h); lens.append(n)

print(hyps[:3])
print("chrF++:", round(sacrebleu.corpus_chrf(hyps, [odia[:20]], word_order=2).score, 1))
print("outputs that hit the 512-token limit:", sum(n >= 512 for n in lens), "of 20")

json.dump({"hyps": hyps, "refs": odia[:20]}, open(f"{OUT}/gemma_en2or_test.json", "w", encoding="utf-8"), ensure_ascii=False)

['ଏହି ଯୋଡ଼ିକାଙ୍କର ନିଜର ସନ୍ତାନଙ୍କ ପାଇଁ ଗୋଟିଏ ଗୃహୀକରଣ ଯୋଜନା କରିବାକୁ ସଙ୍କଳ୍ପ କରିପାରେ।', 'বিভিন্ন ধরণের ব্রেন ড্যামেজ, ট্রমা, লেশন এবং টিউমার দীর্ঘদিন ধরে আচরণকে প্রভাবিত করে এবং কিছু মানসিক ফাংশনে পরিবর্তন ঘটায় তা জানা ছিল।', 'ଏଠାରେ ପେନିନସୁܠܐର ପର୍వତాలుର ସ୍ଥାନୀୟ ସ୍ଥାନରେ ମିଳି ବିଘଟ ହୋଇ ପ୍ଲାଟୋରେ ରୂପାନ୍ତର ହୋଇ, ଏହିରତ୍ତେ ଏଲସୱାର୍ଥ୍ ପର୍వତର ୩୬୦ କିମ୍ ବିରମ୍ତ୍ତୀ ସ୍ଥାନରେ ରୂପାନ୍ତର ହୋଇ ମିନେସୋଟା ଗ୍ଲେସର ଦ୍ୱନ୍ଦ୍ୱରେ ବିଭାଜିତ ହୋଇଥାନ୍ତି।']
chrF++: 21.8
outputs that hit the 512-token limit: 2 of 20


In [20]:
import unicodedata

BLOCKS = {"odia": (0x0B00, 0x0B7F), "bengali": (0x0980, 0x09FF), "devanagari": (0x0900, 0x097F),
          "telugu": (0x0C00, 0x0C7F), "latin": (0x0041, 0x024F)}

def odia_share(text):
    letters = [c for c in text if unicodedata.category(c)[0] in "LM"]
    if not letters: return 0.0
    return sum(0x0B00 <= ord(c) <= 0x0B7F for c in letters) / len(letters)

def main_script(text):
    counts = {k: 0 for k in BLOCKS}
    for c in text:
        for k, (a, b) in BLOCKS.items():
            if a <= ord(c) <= b: counts[k] += 1
    return max(counts, key=counts.get) if any(counts.values()) else "none"

for i, (src, h, ref, n) in enumerate(zip(english[:20], hyps, odia[:20], lens)):
    print(i, f"odia_share={odia_share(h):.2f}", main_script(h), "tokens=" + str(n))

0 odia_share=0.99 odia tokens=50
1 odia_share=0.00 bengali tokens=36
2 odia_share=0.96 odia tokens=136
3 odia_share=1.00 odia tokens=126
4 odia_share=0.87 odia tokens=52
5 odia_share=0.96 odia tokens=56
6 odia_share=1.00 odia tokens=96
7 odia_share=0.00 none tokens=20
8 odia_share=0.90 odia tokens=83
9 odia_share=1.00 odia tokens=83
10 odia_share=0.69 odia tokens=121
11 odia_share=0.15 latin tokens=25
12 odia_share=0.96 odia tokens=56
13 odia_share=0.99 odia tokens=512
14 odia_share=0.00 bengali tokens=26
15 odia_share=1.00 odia tokens=108
16 odia_share=1.00 odia tokens=56
17 odia_share=0.97 odia tokens=86
18 odia_share=1.00 odia tokens=512
19 odia_share=0.96 odia tokens=43


In [21]:
from collections import Counter
import unicodedata

def script_counts(text):
    c = Counter()
    for ch in text:
        if unicodedata.category(ch)[0] == "L":
            try: c[unicodedata.name(ch).split()[0]] += 1
            except ValueError: pass
    return c

for i in [1, 7, 11, 14, 13, 18]:
    print(i, dict(script_counts(hyps[i]).most_common(3)))
    print("EN :", english[i][:150])
    print("OUT:", hyps[i][:250])
    print()

refs = odia[:20]
print("human reference odia_share, min / mean:",
      round(min(odia_share(r) for r in refs), 2), round(sum(odia_share(r) for r in refs) / 20, 2))

ok = [i for i, h in enumerate(hyps) if odia_share(h) >= 0.9]
score = lambda idx: round(sacrebleu.corpus_chrf([hyps[i] for i in idx],
                                                 [[odia[i] for i in idx]], word_order=2).score, 1)
print("chrF++ all 20:", score(range(20)), "| only outputs with share >= 0.9:", score(ok), f"({len(ok)} outputs)")

1 {'BENGALI': 71}
EN : It has been known for a long time that different types of brain damage, traumas, lesions, and tumours affect behaviour and cause changes in some menta
OUT: বিভিন্ন ধরণের ব্রেন ড্যামেজ, ট্রমা, লেশন এবং টিউমার দীর্ঘদিন ধরে আচরণকে প্রভাবিত করে এবং কিছু মানসিক ফাংশনে পরিবর্তন ঘটায় তা জানা ছিল।

7 {'HANGUL': 28}
EN : Willingness of foreign governments to honour these documents is just as widely variable.
OUT: 외국 정부들이 이 문서들을 존중하려는 의지는 그만큼이나 다양하다.

11 {'LATIN': 57, 'ORIYA': 6}
EN : Typically there will be a tuition fee to enroll in these educational programs.
OUT: ଏହି ଶିକ୍ଷାଗ arrege mein enrollment karne ke liye aam taur par ek tuition fee hogi.

14 {'BENGALI': 57, 'TELUGU': 4}
EN : However, most signs are indicated only in Catalan because it is established by law as the first official language.
OUT: যাইহোক, বেশিরভাগ সাইনগুলি কাতালান ভাষায় మాత్రమే নির্দেশিত কারণ এটি আইন অনুসারে প্রথম সরকারী ভাষা হিসাবে প্রতিষ্ঠিত।

13 {'ORIYA': 353, 'DEVANAGARI': 3}
EN : There are als

In [1]:
from collections import Counter
import unicodedata

def script_counts(text):
    c = Counter()
    for ch in text:
        if unicodedata.category(ch)[0] == "L":
            try: c[unicodedata.name(ch).split()[0]] += 1
            except ValueError: pass
    return c

def main_script(text):
    c = script_counts(text)
    return c.most_common(1)[0][0] if c else "NONE"

def share(text, script):          # script = "ORIYA" for Odia, "DEVANAGARI" for Hindi
    c = script_counts(text)
    return c[script] / sum(c.values()) if c else 0.0

In [2]:
dev = get("flores_en_or_dev.json")
dev_pairs = [(e["source"], e["target"]) for e in dev["examples"][:20]]

PROMPTS = {
 "A_plain":  "Translate the following English sentence into Odia. Output only the translation.\n\n{s}",
 "B_script": "Translate the following English sentence into Odia. Use the Odia script (ଓଡ଼ିଆ) only. Output only the translation.\n\n{s}",
}

for name, tpl in PROMPTS.items():
    hy, ln = [], []
    for s, _ in dev_pairs:
        h, n = ask(tpl.format(s=s)); hy.append(h); ln.append(n)
    chrf = sacrebleu.corpus_chrf(hy, [[r for _, r in dev_pairs]], word_order=2).score
    wrong = sum(main_script(h) != "ORIYA" for h in hy)
    pure = sum(share(h, "ORIYA") >= 0.9 for h in hy)
    print(name, "chrF++", round(chrf, 1), "| wrong main script:", wrong, "/20 | share>=0.9:", pure, "/20 | hit limit:", sum(n >= 512 for n in ln))

NameError: name 'get' is not defined

In [3]:
!pip install -q -U transformers accelerate bitsandbytes sacrebleu huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [4]:
import os, json, unicodedata
from collections import Counter
from google.colab import drive
from huggingface_hub import hf_hub_download
import sacrebleu

drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/odia_eval"
os.makedirs(OUT, exist_ok=True)

def get(fname):
    p = hf_hub_download("google/IndicGenBench_flores_in", fname, repo_type="dataset")
    return json.load(open(p, encoding="utf-8"))

d = json.load(open(f"{OUT}/sample200.json", encoding="utf-8"))
english, odia, hindi = d["english"], d["odia"], d["hindi"]
print(len(english), len(odia), len(hindi))

def script_counts(text):
    c = Counter()
    for ch in text:
        if unicodedata.category(ch)[0] == "L":
            try: c[unicodedata.name(ch).split()[0]] += 1
            except ValueError: pass
    return c

def main_script(text):
    c = script_counts(text)
    return c.most_common(1)[0][0] if c else "NONE"

def share(text, script):
    c = script_counts(text)
    return c[script] / sum(c.values()) if c else 0.0

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
200 200 200


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained("google/gemma-4-E4B-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-4-E4B-it", device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                           bnb_4bit_compute_dtype=torch.float16))

def ask(prompt, max_new_tokens=512):
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    new = out[0][ids["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), len(new)

print("ready:", ask("Say hello in one word.")[0])

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

ready: Hello.


In [6]:
import os, json, unicodedata
from collections import Counter
from google.colab import drive
from huggingface_hub import hf_hub_download
#import sacrebleu

drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/odia_eval"
os.makedirs(OUT, exist_ok=True)

def get(fname):
    p = hf_hub_download("google/IndicGenBench_flores_in", fname, repo_type="dataset")
    return json.load(open(p, encoding="utf-8"))

d = json.load(open(f"{OUT}/sample200.json", encoding="utf-8"))
english, odia, hindi = d["english"], d["odia"], d["hindi"]
print(len(english), len(odia), len(hindi))

def script_counts(text):
    c = Counter()
    for ch in text:
        if unicodedata.category(ch)[0] == "L":
            try: c[unicodedata.name(ch).split()[0]] += 1
            except ValueError: pass
    return c

def main_script(text):
    c = script_counts(text)
    return c.most_common(1)[0][0] if c else "NONE"

def share(text, script):
    c = script_counts(text)
    return c[script] / sum(c.values()) if c else 0.0

ModuleNotFoundError: No module named 'sacrebleu'

In [5]:
for n in ["OUT", "get", "english", "odia", "hindi", "tok", "model", "ask", "main_script", "share", "sacrebleu"]:
    print(n, "OK" if n in globals() else "MISSING")

OUT OK
get OK
english OK
odia OK
hindi OK
tok MISSING
model MISSING
ask MISSING
main_script OK
share OK
sacrebleu MISSING


In [14]:
!pip install -q -U transformers accelerate "bitsandbytes>=0.46.1" sacrebleu huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [1]:
   import torch
   print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True Tesla T4


In [2]:
import os, json, unicodedata, torch
from collections import Counter
from google.colab import drive
from huggingface_hub import hf_hub_download
import sacrebleu
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/odia_eval"
os.makedirs(OUT, exist_ok=True)

def get(fname):
    p = hf_hub_download("google/IndicGenBench_flores_in", fname, repo_type="dataset")
    return json.load(open(p, encoding="utf-8"))

d = json.load(open(f"{OUT}/sample200.json", encoding="utf-8"))
english, odia, hindi = d["english"], d["odia"], d["hindi"]

def script_counts(text):
    c = Counter()
    for ch in text:
        if unicodedata.category(ch)[0] == "L":
            try: c[unicodedata.name(ch).split()[0]] += 1
            except ValueError: pass
    return c

def main_script(text):
    c = script_counts(text)
    return c.most_common(1)[0][0] if c else "NONE"

def share(text, script):
    c = script_counts(text)
    return c[script] / sum(c.values()) if c else 0.0

tok = AutoTokenizer.from_pretrained("google/gemma-4-E4B-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-4-E4B-it", device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                           bnb_4bit_compute_dtype=torch.float16))

def ask(prompt, max_new_tokens=512):
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    new = out[0][ids["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), len(new)

print(len(english), len(odia), len(hindi))
print("ready:", ask("Say hello in one word.")[0])

ModuleNotFoundError: No module named 'sacrebleu'

In [1]:
!pip install -U transformers accelerate "bitsandbytes>=0.46.1" sacrebleu huggingface_hub

In [2]:
!pip show sacrebleu bitsandbytes transformers accelerate | grep -E "Name|Version"

Name: sacrebleu
Version: 2.6.0
Name: bitsandbytes
Version: 0.50.2
Name: transformers
Version: 5.23.0
Name: accelerate
Version: 1.16.0


In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True Tesla T4


In [2]:
import os, json, unicodedata, torch
from collections import Counter
from google.colab import drive
from huggingface_hub import hf_hub_download
import sacrebleu
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/odia_eval"
os.makedirs(OUT, exist_ok=True)

def get(fname):
    p = hf_hub_download("google/IndicGenBench_flores_in", fname, repo_type="dataset")
    return json.load(open(p, encoding="utf-8"))

d = json.load(open(f"{OUT}/sample200.json", encoding="utf-8"))
english, odia, hindi = d["english"], d["odia"], d["hindi"]

def script_counts(text):
    c = Counter()
    for ch in text:
        if unicodedata.category(ch)[0] == "L":
            try: c[unicodedata.name(ch).split()[0]] += 1
            except ValueError: pass
    return c

def main_script(text):
    c = script_counts(text)
    return c.most_common(1)[0][0] if c else "NONE"

def share(text, script):
    c = script_counts(text)
    return c[script] / sum(c.values()) if c else 0.0

tok = AutoTokenizer.from_pretrained("google/gemma-4-E4B-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-4-E4B-it", device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                           bnb_4bit_compute_dtype=torch.float16))

def ask(prompt, max_new_tokens=512):
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    new = out[0][ids["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), len(new)

print(len(english), len(odia), len(hindi))
print("ready:", ask("Say hello in one word.")[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

200 200 200
ready: Hello.


In [3]:
dev = get("flores_en_or_dev.json")
dev_pairs = [(e["source"], e["target"]) for e in dev["examples"][:20]]

PROMPTS = {
 "A_plain":  "Translate the following English sentence into Odia. Output only the translation.\n\n{s}",
 "B_script": "Translate the following English sentence into Odia. Use the Odia script (ଓଡ଼ିଆ) only. Output only the translation.\n\n{s}",
}

dev_results = {}
for name, tpl in PROMPTS.items():
    hy, ln = [], []
    for s, _ in dev_pairs:
        h, n = ask(tpl.format(s=s)); hy.append(h); ln.append(n)
    chrf = sacrebleu.corpus_chrf(hy, [[r for _, r in dev_pairs]], word_order=2).score
    wrong = sum(main_script(h) != "ORIYA" for h in hy)
    pure = sum(share(h, "ORIYA") >= 0.9 for h in hy)
    hit = sum(n >= 512 for n in ln)
    dev_results[name] = {"chrf": round(chrf, 1), "wrong_script": wrong, "pure": pure, "hit_limit": hit}
    print(name, dev_results[name])

json.dump(dev_results, open(f"{OUT}/prompt_dev_results.json", "w"), indent=2)

flores_en_or_dev.json:   0%|          | 0.00/614k [00:00<?, ?B/s]

A_plain {'chrf': 21.1, 'wrong_script': 5, 'pure': 9, 'hit_limit': 1}
B_script {'chrf': 20.7, 'wrong_script': 1, 'pure': 13, 'hit_limit': 4}


In [4]:
FROZEN = {
 "en2or": "Translate the following English sentence into Odia. Use the Odia script (ଓଡ଼ିଆ) only. Output only the translation.\n\n{s}",
 "en2hi": "Translate the following English sentence into Hindi. Use the Devanagari script (हिन्दी) only. Output only the translation.\n\n{s}",
 "or2en": "Translate the following Odia sentence into English. Output only the translation.\n\n{s}",
 "hi2en": "Translate the following Hindi sentence into English. Output only the translation.\n\n{s}",
}
json.dump(FROZEN, open(f"{OUT}/frozen_prompts.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

In [5]:
import time
def run(task, sources, refs, outfile):
    done = set()
    if os.path.exists(outfile):
        done = {json.loads(l)["i"] for l in open(outfile, encoding="utf-8")}
    t0 = time.time()
    with open(outfile, "a", encoding="utf-8") as f:
        for i, s in enumerate(sources):
            if i in done: continue
            h, n = ask(FROZEN[task].format(s=s))
            f.write(json.dumps({"i": i, "task": task, "src": s, "hyp": h, "ref": refs[i], "ntok": n},
                               ensure_ascii=False) + "\n")
            f.flush()
            if (i + 1) % 20 == 0:
                print(task, i + 1, "done,", round(time.time() - t0), "s")

In [4]:
for n in ["OUT", "english", "odia", "tok", "model", "ask", "main_script", "share", "sacrebleu"]:
    print(n, "OK" if n in globals() else "MISSING")

OUT MISSING
english MISSING
odia MISSING
tok MISSING
model MISSING
ask MISSING
main_script MISSING
share MISSING
sacrebleu MISSING


In [5]:
!pip show sacrebleu bitsandbytes transformers accelerate | grep -E "Name|Version"

Name: transformers
Version: 5.22.1
Name: accelerate
Version: 1.15.0


In [1]:
  !pip install -U transformers accelerate "bitsandbytes>=0.46.1" sacrebleu huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 128.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 50.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 83.3 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.15.0
    Uninstalling accelerate-1.15.0:
      Successfully uninstalled accelerate-1.15.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [accelerate]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency con

In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True Tesla T4


In [3]:
import os, json, time, unicodedata, torch
from collections import Counter
from google.colab import drive
from huggingface_hub import hf_hub_download
import sacrebleu
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/odia_eval"
os.makedirs(OUT, exist_ok=True)

def get(fname):
    p = hf_hub_download("google/IndicGenBench_flores_in", fname, repo_type="dataset")
    return json.load(open(p, encoding="utf-8"))

d = json.load(open(f"{OUT}/sample200.json", encoding="utf-8"))
english, odia, hindi = d["english"], d["odia"], d["hindi"]

def script_counts(text):
    c = Counter()
    for ch in text:
        if unicodedata.category(ch)[0] == "L":
            try: c[unicodedata.name(ch).split()[0]] += 1
            except ValueError: pass
    return c

def main_script(text):
    c = script_counts(text)
    return c.most_common(1)[0][0] if c else "NONE"

def share(text, script):
    c = script_counts(text)
    return c[script] / sum(c.values()) if c else 0.0

tok = AutoTokenizer.from_pretrained("google/gemma-4-E4B-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-4-E4B-it", device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                           bnb_4bit_compute_dtype=torch.float16))

def ask(prompt, max_new_tokens=512):
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    new = out[0][ids["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), len(new)

print(len(english), len(odia), len(hindi))
print("ready:", ask("Say hello in one word.")[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

200 200 200
ready: Hello.


In [7]:
p = f"{OUT}/gemma_en2or_200.jsonl"
print(sum(1 for _ in open(p, encoding="utf-8")) if os.path.exists(p) else "no file yet")

78


In [12]:
import os, json, time

FROZEN = {
 "en2or": "Translate the following English sentence into Odia. Use the Odia script (ଓଡ଼ିଆ) only. Output only the translation.\n\n{s}",
 "en2hi": "Translate the following English sentence into Hindi. Use the Devanagari script (हिन्दी) only. Output only the translation.\n\n{s}",
 "or2en": "Translate the following Odia sentence into English. Output only the translation.\n\n{s}",
 "hi2en": "Translate the following Hindi sentence into English. Output only the translation.\n\n{s}",
}
json.dump(FROZEN, open(f"{OUT}/frozen_prompts.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

def run_task(task, sources, refs, outfile):
    done = set()
    if os.path.exists(outfile):
        done = {json.loads(l)["i"] for l in open(outfile, encoding="utf-8")}
    t0 = time.time()
    with open(outfile, "a", encoding="utf-8") as f:
        for i, s in enumerate(sources):
            if i in done: continue
            h, n = ask(FROZEN[task].format(s=s))
            f.write(json.dumps({"i": i, "task": task, "src": s, "hyp": h, "ref": refs[i], "ntok": n},
                               ensure_ascii=False) + "\n")
            f.flush()
            if (i + 1) % 20 == 0:
                print(task, i + 1, "done,", round(time.time() - t0), "s")
    print(task, "finished")

In [13]:
for n in ["OUT", "english", "odia", "ask", "run_task", "FROZEN"]:
    print(n, "OK" if n in globals() else "MISSING")

OUT OK
english OK
odia OK
ask OK
run_task OK
FROZEN OK


In [14]:
run_task("en2or", english[:100], odia, f"{OUT}/gemma_en2or_200.jsonl")

en2or 80 done, 99 s
en2or 100 done, 556 s
en2or finished


In [15]:
p = f"{OUT}/gemma_en2or_200.jsonl"
rows = [json.loads(l) for l in open(p, encoding="utf-8")]
ids = [r["i"] for r in rows]
print("lines:", len(rows), "| unique sentence ids:", len(set(ids)), "| max id:", max(ids))
print("empty outputs:", sum(r["hyp"] == "" for r in rows))
print("mean tokens per output:", round(sum(r["ntok"] for r in rows) / len(rows)))
for r in rows[:3] + rows[-2:]:
    print(r["i"], "|", r["src"][:80]); print("  ->", r["hyp"][:120])

lines: 100 | unique sentence ids: 100 | max id: 99
empty outputs: 0
mean tokens per output: 145
0 | These couples may choose to make an adoption plan for their baby.
  -> ଏହି ଯୋଡ଼ିକାଙ୍କର ନିଜର ସନ୍ତାନଙ୍କ ପାଇଁ ଗୋଟିଏ ଗୃహୀକରଣ ଯୋଜନା କରିବାକୁ ସ୍ଙ୍କଳ୍ପ କରିପାରେ।
1 | It has been known for a long time that different types of brain damage, traumas,
  -> ଭିନ୍ର ବିଭିନ୍ର ପ୍ରକାରର ମସ୍ତିଷ୍କ କ୍ଷତି, ଆଘାତ, ଆଘାତୀୟ ରୂପ ଏବଂ ଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗ
2 | The mountains of the Peninsula here merge into the plateau, then re-emerge to fo
  -> ଏଠାରେ ପେନିନସୁଲାର ପର୍ଯ୍ୟାଙ୍ଗଗୁଡ଼ିକ ପ୍ଲାଟୋରେ ମିଳିଭାସେ, ଏపైରକୁ ଆସି ଏଲସୱାର୍ଥ୍ ପର୍వତର ୩୬୦ କିମ୍ ବିରମ୍ତ୍ତୀ ରୂପାକୃତିର ସୃଷ୍ଟି କରନ
98 | Around the old Medina it's easy to find places selling traditional Moroccan good
  -> ପୁରୀର ପାଖରେ సాంప్రదాయ ମରୋକୀ ସାମଗ୍ରୀ ବିକ୍ରାଳୀ ସ୍ଥାନଗୁଡ଼ିକ ପାଇଁ ସହଜରେ ମିଳେ, ଯେପରିକି טାଗାଇନ୍, ପୋଟ୍ରେଇ, ଚର୍ମୀୟ ସାମଗ୍ରୀ, ହୁକା
99 | Any amendments or requests though should be coursed through the travel agent fir
  -> କୌଣସି ସଂଶୋଧନ କିମ୍ବା ଅନ

In [16]:
import numpy as np, random
from collections import Counter

rows = sorted([json.loads(l) for l in open(p, encoding="utf-8")], key=lambda r: r["i"])
rows = [r for r in rows if r["i"] < 100]
hyps = [r["hyp"] for r in rows]; refs = [r["ref"] for r in rows]
n = len(rows); print(n, "sentences")

chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score
rng = random.Random(0); boots = []
for _ in range(1000):
    idx = [rng.randrange(n) for _ in range(n)]
    boots.append(sacrebleu.corpus_chrf([hyps[i] for i in idx], [[refs[i] for i in idx]], word_order=2).score)
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"chrF++ {chrf:.1f}  (95% bootstrap range {lo:.1f} to {hi:.1f})")

mains = [main_script(h) for h in hyps]
shares = [share(h, "ORIYA") for h in hyps]
loops = sum(r["ntok"] >= 512 for r in rows)
print("main script counts:", Counter(mains).most_common())
print("wrong main script:", sum(m != "ORIYA" for m in mains), "/", n)
print("Odia but mixed (share < 0.9):", sum(m == "ORIYA" and s < 0.9 for m, s in zip(mains, shares)), "/", n)
print("pure Odia (share >= 0.9):", sum(s >= 0.9 for s in shares), "/", n)
print("hit 512-token limit (likely loops):", loops, "/", n)

100 sentences
chrF++ 23.8  (95% bootstrap range 21.0 to 26.6)
main script counts: [('ORIYA', 100)]
wrong main script: 0 / 100
Odia but mixed (share < 0.9): 15 / 100
pure Odia (share >= 0.9): 85 / 100
hit 512-token limit (likely loops): 15 / 100


In [17]:
run_task("en2hi", english[:100], hindi, f"{OUT}/gemma_en2hi.jsonl")

en2hi 20 done, 101 s
en2hi 40 done, 212 s
en2hi 60 done, 314 s
en2hi 80 done, 420 s
en2hi 100 done, 523 s
en2hi finished


In [18]:
import numpy as np, random
from collections import Counter

def score_task(path, target_script, n_max=100, boots=1000):
    rows = sorted([json.loads(l) for l in open(path, encoding="utf-8")], key=lambda r: r["i"])
    rows = [r for r in rows if r["i"] < n_max]
    hyps = [r["hyp"] for r in rows]; refs = [r["ref"] for r in rows]; n = len(rows)
    ids = [r["i"] for r in rows]
    print(path.split("/")[-1], "| lines:", n, "| unique ids:", len(set(ids)),
          "| empty outputs:", sum(h == "" for h in hyps))

    chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score
    rng = random.Random(0); bs = []
    for _ in range(boots):
        idx = [rng.randrange(n) for _ in range(n)]
        bs.append(sacrebleu.corpus_chrf([hyps[i] for i in idx], [[refs[i] for i in idx]], word_order=2).score)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"chrF++ {chrf:.1f}  (95% bootstrap range {lo:.1f} to {hi:.1f})")

    mains = [main_script(h) for h in hyps]
    shares = [share(h, target_script) for h in hyps]
    print("main script counts:", Counter(mains).most_common())
    print("wrong main script:", sum(m != target_script for m in mains), "/", n)
    print("target script but mixed (share < 0.9):",
          sum(m == target_script and s < 0.9 for m, s in zip(mains, shares)), "/", n)
    print("hit 512-token limit (likely loops):", sum(r["ntok"] >= 512 for r in rows), "/", n)
    print("mean output tokens:", round(sum(r["ntok"] for r in rows) / n))
    return rows

hi_rows = score_task(f"{OUT}/gemma_en2hi.jsonl", "DEVANAGARI")

gemma_en2hi.jsonl | lines: 100 | unique ids: 100 | empty outputs: 0
chrF++ 52.8  (95% bootstrap range 49.6 to 55.9)
main script counts: [('DEVANAGARI', 98), ('LATIN', 2)]
wrong main script: 2 / 100
target script but mixed (share < 0.9): 3 / 100
hit 512-token limit (likely loops): 0 / 100
mean output tokens: 32


In [19]:
or_rows = score_task(f"{OUT}/gemma_en2or_200.jsonl", "ORIYA")

gemma_en2or_200.jsonl | lines: 100 | unique ids: 100 | empty outputs: 0
chrF++ 23.8  (95% bootstrap range 21.0 to 26.6)
main script counts: [('ORIYA', 100)]
wrong main script: 0 / 100
target script but mixed (share < 0.9): 15 / 100
hit 512-token limit (likely loops): 15 / 100
mean output tokens: 145


In [20]:
run_task("or2en", odia[:100], english, f"{OUT}/gemma_or2en.jsonl")

or2en 20 done, 76 s
or2en 40 done, 161 s
or2en 60 done, 242 s
or2en 80 done, 324 s
or2en 100 done, 407 s
or2en finished


In [21]:
run_task("hi2en", hindi[:100], english, f"{OUT}/gemma_hi2en.jsonl")

hi2en 20 done, 77 s
hi2en 40 done, 157 s
hi2en 60 done, 237 s
hi2en 80 done, 318 s
hi2en 100 done, 396 s
hi2en finished


In [22]:
tasks = {
  "en2or": (f"{OUT}/gemma_en2or_200.jsonl", "ORIYA"),
  "en2hi": (f"{OUT}/gemma_en2hi.jsonl",     "DEVANAGARI"),
  "or2en": (f"{OUT}/gemma_or2en.jsonl",     "LATIN"),
  "hi2en": (f"{OUT}/gemma_hi2en.jsonl",     "LATIN"),
}
all_rows = {}
for name, (path, script) in tasks.items():
    print("=" * 60, name)
    all_rows[name] = score_task(path, script)

============================================================ en2or
gemma_en2or_200.jsonl | lines: 100 | unique ids: 100 | empty outputs: 0
chrF++ 23.8  (95% bootstrap range 21.0 to 26.6)
main script counts: [('ORIYA', 100)]
wrong main script: 0 / 100
target script but mixed (share < 0.9): 15 / 100
hit 512-token limit (likely loops): 15 / 100
mean output tokens: 145
============================================================ en2hi
gemma_en2hi.jsonl | lines: 100 | unique ids: 100 | empty outputs: 0
chrF++ 52.8  (95% bootstrap range 49.6 to 55.9)
main script counts: [('DEVANAGARI', 98), ('LATIN', 2)]
wrong main script: 2 / 100
target script but mixed (share < 0.9): 3 / 100
hit 512-token limit (likely loops): 0 / 100
mean output tokens: 32
============================================================ or2en
gemma_or2en.jsonl | lines: 100 | unique ids: 100 | empty outputs: 0
chrF++ 52.1  (95% bootstrap range 49.7 to 54.7)
main script counts: [('LATIN', 100)]
wrong main script: 0 / 100
target

In [23]:
a = {r["i"]: r for r in all_rows["or2en"]}
b = {r["i"]: r for r in all_rows["hi2en"]}
common = sorted(set(a) & set(b)); n = len(common)

def chrf_of(rows_dict, idx):
    return sacrebleu.corpus_chrf([rows_dict[i]["hyp"] for i in idx],
                                 [[rows_dict[i]["ref"] for i in idx]], word_order=2).score

rng = random.Random(0); diffs = []
for _ in range(1000):
    idx = [common[rng.randrange(n)] for _ in range(n)]
    diffs.append(chrf_of(b, idx) - chrf_of(a, idx))
d0 = chrf_of(b, common) - chrf_of(a, common)
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"Hindi→En minus Odia→En chrF++: {d0:.1f}  (95% paired bootstrap range {lo:.1f} to {hi:.1f})")

Hindi→En minus Odia→En chrF++: 10.3  (95% paired bootstrap range 6.9 to 13.5)


In [24]:
import csv
summary = []
for name, rows in all_rows.items():
    hy = [r["hyp"] for r in rows]; rf = [r["ref"] for r in rows]
    summary.append({"task": name, "n": len(rows),
                    "chrf": round(sacrebleu.corpus_chrf(hy, [rf], word_order=2).score, 1),
                    "hit_limit": sum(r["ntok"] >= 512 for r in rows)})
with open(f"{OUT}/summary_gemma.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=summary[0].keys()); w.writeheader(); w.writerows(summary)
print(summary)

[{'task': 'en2or', 'n': 100, 'chrf': 23.8, 'hit_limit': 15}, {'task': 'en2hi', 'n': 100, 'chrf': 52.8, 'hit_limit': 0}, {'task': 'or2en', 'n': 100, 'chrf': 52.1, 'hit_limit': 0}, {'task': 'hi2en', 'n': 100, 'chrf': 62.4, 'hit_limit': 0}]


In [25]:
for name, rows in all_rows.items():
    ids = [r["i"] for r in rows]
    print(name, "lines:", len(rows), "| unique ids:", len(set(ids)), "| empty:", sum(r["hyp"] == "" for r in rows))

en2or lines: 100 | unique ids: 100 | empty: 0
en2hi lines: 100 | unique ids: 100 | empty: 0
or2en lines: 100 | unique ids: 100 | empty: 0
hi2en lines: 100 | unique ids: 100 | empty: 0


In [26]:
def cat(r):
    h = r["hyp"]; m = main_script(h); s = share(h, "ORIYA")
    if r["ntok"] >= 512: return "hit limit (loop?)"
    if m != "ORIYA":     return "wrong script: " + m
    if s < 0.9:          return "Odia with mixed scripts"
    return "pure Odia"

rows = all_rows["en2or"]
cats = Counter(cat(r) for r in rows)
print(cats.most_common())

for c in cats:
    sub = [r for r in rows if cat(r) == c]
    score = sacrebleu.corpus_chrf([r["hyp"] for r in sub], [[r["ref"] for r in sub]], word_order=2).score
    print(f"{c:28s} n={len(sub):3d}  chrF++ {score:.1f}")

for r in [r for r in rows if r["ntok"] >= 512][:3]:
    w = r["hyp"].split()
    print("unique-word ratio:", round(len(set(w)) / len(w), 2), "|", r["hyp"][:120])

[('pure Odia', 71), ('hit limit (loop?)', 15), ('Odia with mixed scripts', 14)]
pure Odia                    n= 71  chrF++ 30.1
hit limit (loop?)            n= 15  chrF++ 8.0
Odia with mixed scripts      n= 14  chrF++ 27.1
unique-word ratio: 1.0 | ଭିନ୍ର ବିଭିନ୍ର ପ୍ରକାରର ମସ୍ତିଷ୍କ କ୍ଷତି, ଆଘାତ, ଆଘାତୀୟ ରୂପ ଏବଂ ଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗୁମ୍ଫାଗ
unique-word ratio: 0.17 | ଯାହାଙ୍କର ବିବରଣୀ ଅଛି, ସେହିପରି କିଛି ବିସ୍ଫୋରୀ ବସ୍ ମଧ୍ୟ ହେబ్రୋନ୍ ଆଡ଼କୁ ଯାଉଛନ୍ତି, ଯାହା ବିବର୍ତ୍ତଙ୍କର ବିଧାଙ୍କର ସ୍ଥାନୀୟ ସ୍ଥାନୀୟ 
unique-word ratio: 0.1 | କେକଣିକ ରୋହିତାଙ୍କୁ ରକ୍ଷା କରିବାକୁ ସଫଳ ହୋଇଛନ୍ତି ଏବଂ ସତ୍ତ୍ୱାସୀ ସଂଖ୍ୟାରେ ବିରତ୍ତରେ ବିରତ୍ତରେ ବିରତ୍ତରେ ବିରତ୍ତରେ ବିରତ୍ତରେ ବିରତ୍ତର


In [27]:
import zlib, numpy as np

def comp_ratio(t):
    b = t.encode("utf-8")
    return len(zlib.compress(b)) / max(len(b), 1)      # low value = very repetitive

rows = all_rows["en2or"]
for r in rows: r["cr"] = comp_ratio(r["hyp"])
lim   = [r for r in rows if r["ntok"] >= 512]
nolim = [r for r in rows if r["ntok"] < 512]

print("hit-limit outputs, compression ratio:", np.round(sorted(r["cr"] for r in lim), 2))
print("other outputs: median", round(float(np.median([r["cr"] for r in nolim])), 2),
      "| lowest five:", np.round(sorted(r["cr"] for r in nolim)[:5], 2))

def chrf(rs):
    return sacrebleu.corpus_chrf([r["hyp"] for r in rs], [[r["ref"] for r in rs]], word_order=2).score
print("chrF++ all 100:", round(chrf(rows), 1), "| without the 15 limit-hitters:", round(chrf(nolim), 1))

hit-limit outputs, compression ratio: [0.02 0.02 0.03 0.03 0.06 0.06 0.06 0.07 0.07 0.07 0.08 0.09 0.09 0.1
 0.11]
other outputs: median 0.53 | lowest five: [0.42 0.42 0.44 0.44 0.46]
chrF++ all 100: 23.8 | without the 15 limit-hitters: 29.7


In [28]:
import csv, random
rng = random.Random(1)
sample = rng.sample(rows, 30)
with open(f"{OUT}/human_rating_en2or.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["id", "english_source", "model_output_odia", "rating_1to5", "notes"])
    for r in sample:
        w.writerow([r["i"], r["src"], r["hyp"][:600], "", ""])
print("saved", len(sample))

saved 30


In [29]:
# 1) sanity: every task should show 100 lines, 100 unique ids, 0 empty
for name, rows in all_rows.items():
    ids = [r["i"] for r in rows]
    print(name, "lines:", len(rows), "| unique ids:", len(set(ids)), "| empty:", sum(r["hyp"] == "" for r in rows))

# 2) is Odia→English really worse than Hindi→English? (paired bootstrap, same references)
a = {r["i"]: r for r in all_rows["or2en"]}
b = {r["i"]: r for r in all_rows["hi2en"]}
common = sorted(set(a) & set(b)); n = len(common)

def chrf_of(d, idx):
    return sacrebleu.corpus_chrf([d[i]["hyp"] for i in idx], [[d[i]["ref"] for i in idx]], word_order=2).score

rng = random.Random(0); diffs = []
for _ in range(1000):
    idx = [common[rng.randrange(n)] for _ in range(n)]
    diffs.append(chrf_of(b, idx) - chrf_of(a, idx))
d0 = chrf_of(b, common) - chrf_of(a, common)
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"Hindi→En minus Odia→En chrF++: {d0:.1f}  (95% paired bootstrap range {lo:.1f} to {hi:.1f})")

en2or lines: 100 | unique ids: 100 | empty: 0
en2hi lines: 100 | unique ids: 100 | empty: 0
or2en lines: 100 | unique ids: 100 | empty: 0
hi2en lines: 100 | unique ids: 100 | empty: 0
Hindi→En minus Odia→En chrF++: 10.3  (95% paired bootstrap range 6.9 to 13.5)


In [30]:
import os, json
pub = f"{OUT}/public"; os.makedirs(pub, exist_ok=True)
for name in ["en2or_200", "en2hi", "or2en", "hi2en"]:
    src = f"{OUT}/gemma_{name}.jsonl"
    if not os.path.exists(src): print("missing:", src); continue
    with open(src, encoding="utf-8") as f, open(f"{pub}/gemma_{name}_outputs.jsonl", "w", encoding="utf-8") as g:
        for l in f:
            r = json.loads(l)
            g.write(json.dumps({"i": r["i"], "task": r["task"], "hyp": r["hyp"], "ntok": r["ntok"]}, ensure_ascii=False) + "\n")
print(os.listdir(pub))

['gemma_en2or_200_outputs.jsonl', 'gemma_en2hi_outputs.jsonl', 'gemma_or2en_outputs.jsonl', 'gemma_hi2en_outputs.jsonl']


In [8]:
ex = en_or["examples"][0]
print(list(ex.keys()))
for k, v in ex.items():
    print(k, "->", str(v)[:100])

N = 200
english = [e["source"] for e in en_or["examples"][:N]]
odia    = [e["target"] for e in en_or["examples"][:N]]
hindi   = [e["target"] for e in en_hi["examples"][:N]]
print(english[0]); print(odia[0]); print(hindi[0])

['source', 'target', 'lang', 'translation_direction']
source -> Taiwan beginning start way back in 15th century where European sailors passing by record the island’
target -> ତାଇୱାନ 15 ତମ ଶତାବ୍ଦୀର ଆରମ୍ଭରେ ଆରମ୍ଭ କରିଥିଲା ଯେଉଁଠାରେ ୟୁରୋପୀୟ ନାବିକମାନେ ଦ୍ବୀପପୁଞ୍ଜର ନାମ ଇଲହା ଫର୍ମୋସା 
lang -> or
translation_direction -> enxx
Taiwan beginning start way back in 15th century where European sailors passing by record the island’s name as Ilha Formosa, or beautiful island.
ତାଇୱାନ 15 ତମ ଶତାବ୍ଦୀର ଆରମ୍ଭରେ ଆରମ୍ଭ କରିଥିଲା ଯେଉଁଠାରେ ୟୁରୋପୀୟ ନାବିକମାନେ ଦ୍ବୀପପୁଞ୍ଜର ନାମ ଇଲହା ଫର୍ମୋସା କିମ୍ବା ସୁନ୍ଦର ଦ୍ବୀପ ଭାବରେ ରେକର୍ଡ କରିଥିଲେ।
मध्यावधि चुनाव में एक हफ्ते तक हारने के बाद, बुश ने लोगों को एशिया में व्यापार के विस्तार के बारे में बताया।
